In [10]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_frames_grid(
    frames,                      # torch.Tensor 或 np.ndarray, 形状 [H, W, T]
    epoch: int,
    rows: int = 4,
    cols: int = 5,
    cmap: str = "viridis",
    cbar_mode: str = "single",   # "single" | "each" | "none"
    share_scale: bool = True,    # True: 所有子图同一颜色范围；False: 各自范围
    index_base: int = 1,         # 子图标题中的 idx 起始值
    suptitle: str | None = None,
    figsize=(15, 12),
    save_path: str | None = None,
    dpi: int = 160,

    # ---- single 模式相关（沿用你现有逻辑；不会错误挤压子图）----
    grid_wspace_single: float = 0.01,  # single 时（若你用 single）主图横向间距
    cbar_col_ratio: float = 0.03,      # single 时右侧整列色条的相对宽度

    # ---- each 模式：显式增大横向间距，避免色条与右侧子图重叠 ----
    grid_wspace_each: float = 0.15,    # each 时增大子图横向间距（越大越宽）
    cbar_size_each: str = "4%",        # each 时每个色条自身宽度
    cbar_pad_each: float = 0.02,       # each 时色条相对该子图的间隔
):
    """
    将 [H, W, T] 的帧以 rows x cols 网格绘制。
    - cbar_mode="single": 统一色条放在最右一整列；默认不会遮挡任何子图。
    - cbar_mode="each":   每个子图右侧一个色条；**会自动加大横向间距**，避免与右侧子图重叠。
    - cbar_mode="none":   不画色条。
    - share_scale: 是否共享颜色范围（single 会被强制共享）。
    """
    # 预处理输入
    if isinstance(frames, torch.Tensor):
        frames_np = frames.detach().cpu().numpy()
    else:
        frames_np = np.asarray(frames)
    if frames_np.ndim != 3:
        raise ValueError(f"`frames` 需要是 [H, W, T]，但得到 {frames_np.shape}")

    H, W, T = frames_np.shape
    if rows * cols < T:
        raise ValueError(f"网格容量不足：rows*cols={rows*cols} < T={T}")

    # single -> 强制共享范围
    if cbar_mode == "single" and not share_scale:
        share_scale = True

    # 颜色范围
    global_vmin = np.nanmin(frames_np) if share_scale else None
    global_vmax = np.nanmax(frames_np) if share_scale else None

    # =============== 创建画布与坐标轴 ===============
    if cbar_mode == "single":
        # 右侧留出整列给统一色条
        fig = plt.figure(figsize=figsize, constrained_layout=True)
        gs = fig.add_gridspec(rows, cols + 1, width_ratios=[1]*cols + [cbar_col_ratio])
        axs = np.array([[fig.add_subplot(gs[r, c]) for c in range(cols)] for r in range(rows)])
        cax = fig.add_subplot(gs[:, -1])
        try:
            fig.set_constrained_layout_pads(wspace=grid_wspace_single)
        except Exception:
            pass
    else:
        # each / none：常规网格
        fig, axs = plt.subplots(rows, cols, figsize=figsize, constrained_layout=True)
        axs = np.array(axs).reshape(rows, cols)
        cax = None
        if cbar_mode == "each":
            # —— 关键：ONLY 在 each 模式下加大横向间距 —— #
            try:
                fig.set_constrained_layout_pads(wspace=grid_wspace_each)
            except Exception:
                pass

    # =============== 绘图 ===============
    ims = []
    for i in range(T):
        r, c = divmod(i, cols)
        ax = axs[r, c]
        if share_scale:
            im = ax.imshow(frames_np[:, :, i], cmap=cmap, vmin=global_vmin, vmax=global_vmax)
        else:
            im = ax.imshow(frames_np[:, :, i], cmap=cmap)

        ax.set_title(f"idx={index_base + i} | epoch={epoch}")
        ax.set_xticks([]); ax.set_yticks([])
        ims.append(im)

        # each：每个子图自己的色条（与右邻的间距由 wspace 控制）
        if cbar_mode == "each":
            divider = make_axes_locatable(ax)
            cax_local = divider.append_axes("right", size=cbar_size_each, pad=cbar_pad_each)
            fig.colorbar(im, cax=cax_local)

    # 补齐空白子图
    for j in range(T, rows * cols):
        r, c = divmod(j, cols)
        axs[r, c].axis("off")

    # single：统一色条
    if cbar_mode == "single" and ims:
        fig.colorbar(ims[-1], cax=cax)

    if suptitle:
        fig.suptitle(suptitle, y=0.995)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=dpi)
        plt.close(fig)
    else:
        plt.show()




In [11]:
import torch
import os

root = "modes96_width80_epochs2000_Tin10_T10_batch2_2/adv_datasets_samples"
os.makedirs("fig_out", exist_ok=True)

for epoch in [10, 20, 30, 40, 50, 60, 70]:
    data = torch.load(f"{root}/adv_first_batch_epoch00{epoch}.pt", weights_only=False)
    # 这里建议用 dim 而不是 axis（torch 用 dim）
    dataset = torch.concat([data["x_adv"], data["y"]], dim=-1)  # [1, H, W, T]
    frames = dataset[0]  # [H, W, T]

    # 例1：整幅图一个 colorbar，所有子图共享颜色范围
    plot_frames_grid(
        frames,
        epoch=epoch,
        rows=4,
        cols=5,
        cmap="viridis",
        cbar_mode="single",     # "single" | "each" | "none"
        share_scale=True,       # True: 统一颜色范围；False: 各自范围
        index_base=1,
        save_path=f"fig_out/epoch{epoch:04d}_idx1_single_shared.png"
    )

    # 例2：每个子图都有自己的 colorbar，且各自独立颜色范围
    plot_frames_grid(
        frames,
        epoch=epoch,
        rows=4,
        cols=5,
        cmap="viridis",
        cbar_mode="each",
        share_scale=False,
        index_base=1,
        save_path=f"fig_out/epoch{epoch:04d}_idx1_each_individual.png"
    )

    frames = dataset[1]  # [H, W, T]

    # 例1：整幅图一个 colorbar，所有子图共享颜色范围
    plot_frames_grid(
        frames,
        epoch=epoch,
        rows=4,
        cols=5,
        cmap="viridis",
        cbar_mode="single",     # "single" | "each" | "none"
        share_scale=True,       # True: 统一颜色范围；False: 各自范围
        index_base=2,
        save_path=f"fig_out/epoch{epoch:04d}_idx2_single_shared.png"
    )

    # 例2：每个子图都有自己的 colorbar，且各自独立颜色范围
    plot_frames_grid(
        frames,
        epoch=epoch,
        rows=4,
        cols=5,
        cmap="viridis",
        cbar_mode="each",
        share_scale=False,
        index_base=2,
        save_path=f"fig_out/epoch{epoch:04d}_idx2_each_individual.png"
    )
